# Load Libraries

In [1]:
!pip install -U "transformers>=4.42.3" bitsandbytes accelerate peft

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 16.0 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 0.32.1
    Uninstalling accelerate-0.32.1:
      Successfully uninstalled accelerate-0.32.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.42.3
    Uninstalling transformers-4.42.3:
      Successfully uninstalled transformers-4.42.3


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"

VER=157

# FINAL SOLUTION IS USE_QLORA=FALSE, TRAIN_100_PERCENT=TRUE, ADD_33K=TRUE, DEBUG=FALSE
USE_QLORA = True
TRAIN_100_PERCENT = False
ADD_33K = False
DEBUG = True

In [3]:
import os
import copy
from dataclasses import dataclass

import numpy as np
import torch
from datasets import Dataset
from transformers import (
    BitsAndBytesConfig,
    Gemma2ForSequenceClassification,
    GemmaTokenizerFast,
    Gemma2Config,
    PreTrainedTokenizerBase, 
    EvalPrediction,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import log_loss, accuracy_score

2024-08-14 19:48:26.693176: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-14 19:48:26.693304: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-14 19:48:26.859739: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


# Tokenizer and Custom 3-Head Model
From EDA we see there are 58 unique models in train data after renaming some. So we hardcode the value of 58 in code below.

In [4]:
tokenizer = GemmaTokenizerFast.from_pretrained("/kaggle/input/gemma2-9b-it-fp16")
tokenizer.add_eos_token = True  # We'll add <eos> at the end
tokenizer.padding_side = "right"

In [5]:
import torch
import torch.nn as nn
from transformers import Gemma2ForSequenceClassification, Gemma2Config

class CustomGemma2ForSequenceClassification(Gemma2ForSequenceClassification):
    def __init__(self, config, num_labels_head1=58, num_labels_head2=58):
        super().__init__(config)
        self.num_labels_head1 = num_labels_head1
        self.num_labels_head2 = num_labels_head2
        self.classifier_head1 = nn.Linear(config.hidden_size, num_labels_head1, bias=False)
        self.classifier_head2 = nn.Linear(config.hidden_size, num_labels_head2, bias=False)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        device = input_ids.device

        if labels is not None:
            labels = labels.to(device)
            outputs = super().forward(input_ids, attention_mask=attention_mask, labels=labels[:, 0], output_hidden_states=True)
        else:
            outputs = super().forward(input_ids, attention_mask=attention_mask)

        last_token_indices = (torch.sum(attention_mask, dim=1) - 1).to(device)
        last_token_outputs = outputs.hidden_states[-1].to(device)[
            torch.arange(outputs.hidden_states[-1].shape[0], device=device), last_token_indices]

        outputs_head1 = self.classifier_head1(last_token_outputs).to(device)
        outputs_head2 = self.classifier_head2(last_token_outputs).to(device)

        if labels is not None:
            labels_head1 = labels[:, 1].to(device)
            labels_head2 = labels[:, 2].to(device)
            
            loss_head1 = nn.CrossEntropyLoss()(outputs_head1, labels_head1)
            loss_head2 = nn.CrossEntropyLoss()(outputs_head2, labels_head2)
            loss = outputs.loss.to(device) + 0.1 * loss_head1 + 0.1 * loss_head2
            return {"loss": loss, "logits": (outputs.logits, outputs_head1, outputs_head2)}
        else:
            return {"logits": (outputs.logits, outputs_head1, outputs_head2)}

# Load Gemma2 FP16 Without Quantization

In [6]:
from peft import PeftModel
config2 = Gemma2Config.from_pretrained("/kaggle/input/gemma2-9b-it-fp16")
config2.num_labels = 3
model = CustomGemma2ForSequenceClassification.from_pretrained(
    "/kaggle/input/gemma2-9b-it-fp16",
    config=config2,
    num_labels_head1=58,
    num_labels_head2=58,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.config.use_cache = False
model = PeftModel.from_pretrained(model, "/kaggle/input/16th-place-train-1-of-3/LoRA-v157")
model

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of CustomGemma2ForSequenceClassification were not initialized from the model checkpoint at /kaggle/input/gemma2-9b-it-fp16 and are newly initialized: ['classifier_head1.weight', 'classifier_head2.weight', 'score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): CustomGemma2ForSequenceClassification(
      (model): Gemma2Model(
        (embed_tokens): Embedding(256000, 3584, padding_idx=0)
        (layers): ModuleList(
          (0-41): 42 x Gemma2DecoderLayer(
            (self_attn): Gemma2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=64, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=64, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleD

# Merge Adapter to Gemma2 FP16

In [7]:
model = model.merge_and_unload()
model

CustomGemma2ForSequenceClassification(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2SdpaAttention(
          (q_proj): Linear(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear(in_features=4096, out_features=3584, bias=False)
          (rotary_emb): Gemma2RotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=3584, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): 

# Save Merged Model

In [8]:
model.save_pretrained(f"merged-v{VER}") 
tokenizer.save_pretrained(f"merged-v{VER}")

('merged-v157/tokenizer_config.json',
 'merged-v157/special_tokens_map.json',
 'merged-v157/tokenizer.model',
 'merged-v157/added_tokens.json',
 'merged-v157/tokenizer.json')

In [9]:
del model
torch.cuda.empty_cache()

# Quantize to 8bit (or 4bit)
We will load the merged model from disk, quantize it, and save quantized model to disk.

In [10]:
from transformers import BitsAndBytesConfig
if not USE_QLORA: 
    bnb_config = BitsAndBytesConfig(
        load_in_8bit = True,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules = ["score","classifier_head1", "classifier_head2"]
    )
else: # USE SAME QUANTIZATION THAT WE TRAINED WITH
    bnb_config = BitsAndBytesConfig(
        load_in_4bit = True,
        bnb_4bit_quant_type = "nf4", #nf4 or fp4
        bnb_4bit_use_double_quant = False,
        bnb_4bit_compute_dtype=torch.float16,
        llm_int8_skip_modules = ["score","classifier_head1", "classifier_head2"]
    )

In [11]:
config2 = Gemma2Config.from_pretrained("/kaggle/input/gemma2-9b-it-fp16")
config2.num_labels = 3
model = CustomGemma2ForSequenceClassification.from_pretrained(
    f"merged-v{VER}",
    config=config2,
    num_labels_head1=58,
    num_labels_head2=58,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config = bnb_config,
)
model.config.use_cache = False
model

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

CustomGemma2ForSequenceClassification(
  (model): Gemma2Model(
    (embed_tokens): Embedding(256000, 3584, padding_idx=0)
    (layers): ModuleList(
      (0-41): 42 x Gemma2DecoderLayer(
        (self_attn): Gemma2SdpaAttention(
          (q_proj): Linear4bit(in_features=3584, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (v_proj): Linear4bit(in_features=3584, out_features=2048, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=3584, bias=False)
          (rotary_emb): Gemma2RotaryEmbedding()
        )
        (mlp): Gemma2MLP(
          (gate_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (up_proj): Linear4bit(in_features=3584, out_features=14336, bias=False)
          (down_proj): Linear4bit(in_features=14336, out_features=3584, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma2RMSNorm((3584,), eps=1e-06)
        

# Save Quantized Model

In [12]:
os.system(f"rm -r merged-v{VER}") 

0

In [13]:
if USE_QLORA:
    model.save_pretrained(f"merged-v{VER}-4bit") 
    tokenizer.save_pretrained(f"merged-v{VER}-4bit")
else:
    model.save_pretrained(f"merged-v{VER}-8bit") 
    tokenizer.save_pretrained(f"merged-v{VER}-8bit")